In [1]:
from dataclasses import dataclass
import numpy as np

@dataclass
class Grid:
    rows: int
    cols: int
    step_reward: int
    terminals: dict
    walls: set
    noise: float = 0.0
    actions = dict(up=(-1, 0), right=(0, 1), down=(1, 0), left=(0, -1))
    arrows = {'up': "↑", 'right': "→", 'down': "↓", 'left': "←"}
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return self.terminals[cell]
        elif cell in self.walls:
            return '#'
        else:
            return '·'
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print(''.join([f'{v:>3} ' for v in range(self.cols)]))
                if c == 0:
                    print(f'{r:>3} ', end="")
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def step(self, cell, action):
        result = []
        action_keys = list(self.actions.keys())
        aind = list(self.actions.keys()).index(action)
        n_actions = len(action_keys)
        for a, prob in zip(
            [aind, (aind + 1) % n_actions, (aind - 1) % n_actions],
            [1 - self.noise, self.noise / 2, self.noise / 2],
        ):
            movement = self.actions[action_keys[a]]
            next_cell = (cell[0] + movement[0], cell[1] + movement[1])
            outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
            on_wall = next_cell in self.walls
            if outside or on_wall:
                next_cell = cell

            if next_cell in self.terminals:
                reward = self.terminals[next_cell]
            else:
                reward = self.step_reward
            result.append((prob, next_cell, reward))
        return result
    def properties(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
    noise={self.noise},
)
'''


default_grid = Grid(
    rows=3,
    cols=4,
    step_reward=-0.04,
    terminals={(0, 3): 1},
    walls={(1, 1)},
    noise=0.2,
)
default_grid

Grid(rows=3, cols=4, step_reward=-0.04, terminals={(0, 3): 1}, walls={(1, 1)}, noise=0.2)

In [2]:
default_grid.render()

 r/c  0   1   2   3 
  0   ·   ·   ·   1 
  1   ·   #   ·   · 
  2   ·   ·   ·   · 


In [3]:
class Sampler:
    def __init__(self, env: Grid, rng: np.random.Generator):
        self.env = env
        self.rng = rng
        self.empty_cells = [
            (r, c)
            for r in range(env.rows)
            for c in range(env.cols)
            if (r, c) not in env.terminals and (r, c) not in env.walls
        ]
        self.nS = len(self.empty_cells)

    def reset(self):
        i = self.rng.choice(len(self.empty_cells))
        return self.empty_cells[i]
    def step(self, cell, action):
        result = self.env.step(cell, action)
        i = self.rng.choice(len(result), p=[p for p, _, _ in result])
        _, next_cell, reward = result[i]
        done = True if next_cell in self.env.terminals else False
        return next_cell, reward, done

In [4]:
import io, contextlib, functools

def silent(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        with contextlib.redirect_stdout(io.StringIO()):
            return fn(*args, **kwargs)
    return wrapper

In [5]:
def render_policy(grid: Grid, policy):
    for r in range(len(policy)):
        for c in range(len(policy[0])):
            if r == 0 and c == 0:
                print("r/c", end="")
                print(''.join([f'{v:>2} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>2} ', end="")

            if policy[r][c] != None:
                value = grid.arrows[max(policy[r][c], key=policy[r][c].get)]
            else:
                value = grid.cell_repr(r, c)
            print(f' {value} ', end='')
        print()
    print('------------------')

def show_V(grid: Grid, V):
    for r in range(grid.rows):
        for c in range(grid.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")
            print(f'{round(V[r][c], 2):>4} ', end="")
        print()
    print('------------------')


def value_iteration(grid: Grid, gamma=0.9, theta=1e-6, max_iters=1000):
    print('---------- value_iteration ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                def q(action):
                    result = grid.step(cell, action)
                    q_value = 0
                    for prob, next_cell, reward in result:
                        q_value += prob * (
                            reward + gamma * V_old[next_cell[0]][next_cell[1]]
                        )
                    return q_value
                new_value = float('-inf')
                for action in grid.actions:
                    value = q(action)
                    if value > new_value:
                        new_value = value
                        policy[r][c] = {action: 1.0}
                V[r][c] = new_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        show_V(grid, V)
        i += 1
    render_policy(grid, policy)
    converged = delta <= theta
    if converged:
        print(f'value_iteration converged in {i} iterations')
    else:
        print(f'value_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [6]:
policy, V, converged = value_iteration(default_grid);

---------- value_iteration ------------
  r/c   0    1    2    3 
   0 -0.04 -0.04 0.79    0 
   1 -0.04    0 -0.04 0.79 
   2 -0.04 -0.04 -0.04 -0.04 
------------------
  r/c   0    1    2    3 
   0 -0.08 0.52 0.86    0 
   1 -0.08    0  0.6 0.86 
   2 -0.08 -0.08 -0.08 0.52 
------------------
  r/c   0    1    2    3 
   0 0.32 0.67 0.92    0 
   1 -0.11    0 0.71 0.92 
   2 -0.11 -0.11 0.43 0.62 
------------------
  r/c   0    1    2    3 
   0 0.46 0.75 0.94    0 
   1 0.17    0 0.77 0.94 
   2 -0.14 0.25 0.52 0.72 
------------------
  r/c   0    1    2    3 
   0 0.55 0.77 0.95    0 
   1 0.33    0 0.79 0.95 
   2 0.14 0.38  0.6 0.75 
------------------
  r/c   0    1    2    3 
   0 0.59 0.78 0.95    0 
   1 0.42    0  0.8 0.95 
   2 0.27 0.46 0.63 0.76 
------------------
  r/c   0    1    2    3 
   0 0.61 0.78 0.95    0 
   1 0.46    0  0.8 0.95 
   2 0.35  0.5 0.64 0.77 
------------------
  r/c   0    1    2    3 
   0 0.62 0.78 0.95    0 
   1 0.48    0  0.8 0.95 
   2

In [7]:
def read_policy(grid: Grid, V, gamma=0.9, incumbent_policy=None):
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            cell = (r, c)
            if cell in grid.walls or cell in grid.terminals:
                continue
            def q(action):
                result = grid.step(cell, action)
                q_value = 0
                for prob, next_cell, reward in result:
                    q_value += prob * (
                        reward + gamma * V[next_cell[0]][next_cell[1]]
                    )
                return q_value
            new_action = max(grid.actions, key=q)
            if incumbent_policy:
                incumbent_action = max(incumbent_policy[r][c], key=incumbent_policy[r][c].get)
                if q(new_action) - q(incumbent_action) < 1e-9:
                    new_action = incumbent_action
            policy[r][c] = {new_action: 1.0}
    return policy


policy = read_policy(default_grid, V)
policy

[[{'right': 1.0}, {'right': 1.0}, {'right': 1.0}, None],
 [{'up': 1.0}, None, {'up': 1.0}, {'up': 1.0}],
 [{'right': 1.0}, {'right': 1.0}, {'up': 1.0}, {'up': 1.0}]]

In [8]:
render_policy(default_grid, policy)

r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------


In [9]:
def make_Q(grid: Grid):
    """Q[r][c] is a {action_name: value} dict -- same shape as `policy`.

    All four actions must exist and start EQUAL: control has to be able to
    compare them, and a missing key can never be chosen or learned.
    """
    return [[{a: 0.0 for a in grid.actions} for _ in range(grid.cols)]
            for _ in range(grid.rows)]

In [10]:
def epsilon_greedy(Q, cell, eps, actions, rng):
    """eps-greedy over the Q row at `cell`. Returns an action NAME.

    Random action w.p. eps (EXPLORE), else argmax_a Q[cell][a] (EXPLOIT) with a
    RANDOM tie-break -- at init every action is 0.0, so a fixed tie-break would
    march the agent one direction out of every unexplored cell.
    """
    if rng.random() < eps:
        return actions[int(rng.integers(len(actions)))]
    q = Q[cell[0]][cell[1]]
    best = max(q.values())
    ties = [a for a, v in q.items() if v == best]
    return ties[int(rng.integers(len(ties)))]

In [11]:
ACTIONS = list(default_grid.actions)

Q = make_Q(default_grid)
Q[0][0]['right'] = 1.0          # make 'right' the unique greedy action at (0,0)
rng = np.random.default_rng(1)

n = 40000
print(" eps  | P(greedy 'right') | formula (1-eps)+eps/nA")
print("------+-------------------+-----------------------")
for eps in (0.0, 0.1, 0.3, 1.0):
    picks = [epsilon_greedy(Q, (0, 0), eps, ACTIONS, rng) for _ in range(n)]
    print(f" {eps:.1f}  |      {picks.count('right') / n:.4f}       |"
          f"        {(1 - eps) + eps / len(ACTIONS):.4f}")

# untouched cell: all four tie at 0.0, so the tie-break must spread UNIFORMLY.
# Lock onto one action here and most (cell, action) pairs never get data at all.
picks = [epsilon_greedy(make_Q(default_grid), (1, 0), 0.0, ACTIONS, rng)
         for _ in range(n)]
print("\nuntouched cell, eps=0 ->",
      {a: round(picks.count(a) / n, 3) for a in ACTIONS})

 eps  | P(greedy 'right') | formula (1-eps)+eps/nA
------+-------------------+-----------------------
 0.0  |      1.0000       |        1.0000
 0.1  |      0.9247       |        0.9250
 0.3  |      0.7763       |        0.7750
 1.0  |      0.2485       |        0.2500

untouched cell, eps=0 -> {'up': 0.252, 'right': 0.246, 'down': 0.248, 'left': 0.255}


In [12]:
class ControlSampler(Sampler):
    """Sampler whose reset() returns a FIXED start cell.

    Prediction could use exploring starts to get coverage for free -- that's what
    `Sampler.reset()` does. In control the agent starts where the task starts, and
    eps-greedy is what now has to reach the rest of the grid.
    """

    def __init__(self, env: Grid, rng: np.random.Generator, start=(2, 0)):
        super().__init__(env, rng)
        self.start = start

    def reset(self):
        return self.start

In [13]:
def sarsa(sampler, actions, rng, gamma=0.9, num_episodes=20000, alpha=0.05,
          eps0=1.0, eps_min=0.05, max_steps=1000):
    """SARSA -- ON-policy TD control. Uses only sampler.reset()/step(), never grid.step().

    The update consumes exactly (S, A, R, S', A') -- hence the name:

        Q(s,a) <- Q(s,a) + alpha * [ r + gamma*Q(s',a') - Q(s,a) ]

    a' is drawn ONCE by epsilon_greedy and then reused as the next iteration's
    action. That reuse is the whole on-policy story: the action it bootstraps off
    is the action it actually goes on to take, so SARSA evaluates the eps-greedy
    policy it is FOLLOWING, exploration included -- not the greedy one.

    eps anneals 1.0 -> 0.05, so behaviour starts as a random walk (coverage) and
    ends near-greedy (which is why max_a Q converges toward V*).
    """
    grid = sampler.env
    Q = make_Q(grid)
    for ep in range(num_episodes):
        eps = max(eps_min, eps0 * (1 - ep / num_episodes))
        cell = sampler.reset()
        action = epsilon_greedy(Q, cell, eps, actions, rng)
        for _ in range(max_steps):
            next_cell, reward, done = sampler.step(cell, action)
            next_action = epsilon_greedy(Q, next_cell, eps, actions, rng)
            # a terminal has no future, so there is nothing to bootstrap off
            bootstrap = 0.0 if done else gamma * Q[next_cell[0]][next_cell[1]][next_action]
            td_error = reward + bootstrap - Q[cell[0]][cell[1]][action]
            Q[cell[0]][cell[1]][action] += alpha * td_error
            cell, action = next_cell, next_action     # reuse a' -- this is the "on-policy"
            if done:
                break
    return Q

In [14]:
def policy_from_Q(grid: Grid, Q):
    """Greedy policy read straight off Q -- argmax_a Q[cell][a], no model needed.
    Compare with `read_policy`, which needs grid.step() to do the same job from V."""
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            if (r, c) in grid.walls or (r, c) in grid.terminals:
                continue
            policy[r][c] = {max(Q[r][c], key=Q[r][c].get): 1.0}
    return policy


def V_from_Q(grid: Grid, Q):
    """V(s) = max_a Q(s,a) -- only equals V* once the policy has annealed to greedy."""
    return [[0.0 if (r, c) in grid.walls or (r, c) in grid.terminals
             else max(Q[r][c].values())
             for c in range(grid.cols)] for r in range(grid.rows)]

In [15]:
rng = np.random.default_rng(0)
sampler = ControlSampler(default_grid, np.random.default_rng(0))
Q_sarsa = sarsa(sampler, ACTIONS, rng)

pol_sarsa = policy_from_Q(default_grid, Q_sarsa)
V_sarsa = V_from_Q(default_grid, Q_sarsa)

print("SARSA greedy policy:")
render_policy(default_grid, pol_sarsa)
print("optimal policy (value iteration):")
render_policy(default_grid, policy)

inner = [(r, c) for r in range(default_grid.rows) for c in range(default_grid.cols)
         if (r, c) not in default_grid.walls and (r, c) not in default_grid.terminals]
match = sum(max(pol_sarsa[r][c], key=pol_sarsa[r][c].get)
            == max(policy[r][c], key=policy[r][c].get) for r, c in inner)
print(f"actions matching pi*: {match}/{len(inner)}\n")

print(" cell  | SARSA max_a Q |   V*    |  diff")
print("-------+---------------+---------+--------")
for r, c in inner:
    print(f" ({r},{c}) |    {V_sarsa[r][c]:+.4f}    | {V[r][c]:+.4f} |"
          f" {V_sarsa[r][c] - V[r][c]:+.4f}")
rmse = np.sqrt(np.mean([(V_sarsa[r][c] - V[r][c]) ** 2 for r, c in inner]))
print(f"\nRMSE vs V* = {rmse:.4f}")

SARSA greedy policy:
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
optimal policy (value iteration):
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
actions matching pi*: 10/10

 cell  | SARSA max_a Q |   V*    |  diff
-------+---------------+---------+--------
 (0,0) |    +0.6037    | +0.6267 | -0.0230
 (0,1) |    +0.7829    | +0.7850 | -0.0021
 (0,2) |    +0.9667    | +0.9496 | +0.0171
 (1,0) |    +0.4754    | +0.5015 | -0.0261
 (1,2) |    +0.8061    | +0.8013 | +0.0048
 (1,3) |    +0.9575    | +0.9496 | +0.0079
 (2,0) |    +0.4144    | +0.4212 | -0.0067
 (2,1) |    +0.5307    | +0.5252 | +0.0056
 (2,2) |    +0.6708    | +0.6537 | +0.0172
 (2,3) |    +0.7881    | +0.7720 | +0.0161

RMSE vs V* = 0.0149


In [16]:
def qlearning(sampler, actions, rng, gamma=0.9, num_episodes=20000, alpha=0.05,
          eps0=1.0, eps_min=0.05, max_steps=1000):
    grid = sampler.env
    Q = make_Q(grid)
    for ep in range(num_episodes):
        eps = max(eps_min, eps0 * (1 - ep / num_episodes))
        cell = sampler.reset()
        
        for _ in range(max_steps):
            action = epsilon_greedy(Q, cell, eps, actions, rng)
            next_cell, reward, done = sampler.step(cell, action)
            # a terminal has no future, so there is nothing to bootstrap off
            bootstrap = (
                0.0 if done else gamma * max(Q[next_cell[0]][next_cell[1]].values())
            )
            td_error = reward + bootstrap - Q[cell[0]][cell[1]][action]
            Q[cell[0]][cell[1]][action] += alpha * td_error
            cell = next_cell  
            if done:
                break
    return Q

In [17]:
rng = np.random.default_rng(0)
sampler = ControlSampler(default_grid, np.random.default_rng(0))
Q_qlearning = qlearning(sampler, ACTIONS, rng)

pol_qlearning = policy_from_Q(default_grid, Q_qlearning)
V_qlearning = V_from_Q(default_grid, Q_qlearning)

print("qlearning greedy policy:")
render_policy(default_grid, pol_qlearning)
print("optimal policy (value iteration):")
render_policy(default_grid, policy)

inner = [(r, c) for r in range(default_grid.rows) for c in range(default_grid.cols)
         if (r, c) not in default_grid.walls and (r, c) not in default_grid.terminals]
match = sum(max(pol_qlearning[r][c], key=pol_qlearning[r][c].get)
            == max(policy[r][c], key=policy[r][c].get) for r, c in inner)
print(f"actions matching pi*: {match}/{len(inner)}\n")

print(" cell  | qlearning max_a Q |   V*    |  diff")
print("-------+---------------+---------+--------")
for r, c in inner:
    print(f" ({r},{c}) |    {V_qlearning[r][c]:+.4f}    | {V[r][c]:+.4f} |"
          f" {V_qlearning[r][c] - V[r][c]:+.4f}")
rmse = np.sqrt(np.mean([(V_qlearning[r][c] - V[r][c]) ** 2 for r, c in inner]))
print(f"\nRMSE vs V* = {rmse:.4f}")

qlearning greedy policy:
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
optimal policy (value iteration):
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
actions matching pi*: 10/10

 cell  | qlearning max_a Q |   V*    |  diff
-------+---------------+---------+--------
 (0,0) |    +0.6426    | +0.6267 | +0.0159
 (0,1) |    +0.7802    | +0.7850 | -0.0048
 (0,2) |    +0.9742    | +0.9496 | +0.0246
 (1,0) |    +0.5092    | +0.5015 | +0.0077
 (1,2) |    +0.8043    | +0.8013 | +0.0030
 (1,3) |    +0.9416    | +0.9496 | -0.0080
 (2,0) |    +0.4122    | +0.4212 | -0.0090
 (2,1) |    +0.5367    | +0.5252 | +0.0115
 (2,2) |    +0.6586    | +0.6537 | +0.0049
 (2,3) |    +0.7573    | +0.7720 | -0.0147

RMSE vs V* = 0.0121
